# 05 - Customer Churn Prediction (Classification)

**Objective:** Build and compare classification models for customer churn prediction.

**Target:** 60-day churn (no orders in 60 days)
**Metric:** F1 ≥ 0.70, AUC ≥ 0.80

In [ ]:
import pandas as pd
import numpy as np
import sys
import os

sys.path.insert(0, os.path.abspath('..'))

from src.model_churn import ChurnPredictor
from src.utils import setup_logger

logger = setup_logger(__name__)

print('✅ All imports successful!')

In [ ]:
# Load engineered features
df = pd.read_csv('../data/cleaned/orders_engineered.csv')

print(f'Data shape: {df.shape}')
print(f'Target: 60-day churn')

In [ ]:
# Initialize predictor
predictor = ChurnPredictor(logger=logger)

# Create churn target
df['churn'] = predictor.create_churn_target(df, days_threshold=60)

print(f'\n✅ Churn target created')
print(f'Churn rate: {df["churn"].mean():.1%}')
print(f'Churned customers: {df["churn"].sum():,}')
print(f'Active customers: {(1-df["churn"]).sum():,}')

In [ ]:
# Prepare data
X_train, X_test, y_train, y_test = predictor.prepare_data(df, target='churn')

print(f'\n✅ Data prepared')
print(f'Training set: {X_train.shape[0]:,} samples')
print(f'Test set: {X_test.shape[0]:,} samples')

In [ ]:
# Train all models
models = predictor.train_all_models(X_train, y_train)

print(f'\n✅ All {len(models)} models trained')

In [ ]:
# Evaluate models
results = predictor.evaluate_models(X_test, y_test)

print('\n' + results.to_string())

In [ ]:
# Get confusion matrix
cm = predictor.get_confusion_matrix(X_test, y_test)

print('\n✅ Confusion Matrix:')
print(f'  True Negatives: {cm[0,0]}')
print(f'  False Positives: {cm[0,1]}')
print(f'  False Negatives: {cm[1,0]}')
print(f'  True Positives: {cm[1,1]}')

In [ ]:
# Save best model
os.makedirs('../models/', exist_ok=True)
predictor.save_model('../models/churn_best_model.pkl')

print(f'\n✅ Best model saved')
print(f'Model: {predictor.best_model_name}')

In [ ]:
# Display summary
summary = predictor.get_model_summary()

print('\n MODEL SUMMARY')
for key, value in summary.items():
    print(f'  {key}: {value}')